# 02장 보안 실습 — 환경과 증거 취급


## Goal

합성 출처·KST·partial을 확인하고 작업 사본과 결과를 분리합니다.

[교안과 분석 질문](../../02-bash-setup/02-2-evidence-time.md)을 먼저 읽습니다.


## Setup

Python 커널의 %%bash를 사용합니다. 새 임시 폴더에 합성 자료와 결과 경로를 준비합니다. 외부 접속·서비스 등록·원본 서버 조사는 하지 않습니다. 코드를 검토하고 Setup부터 순서대로 실행합니다. Bash 셀 사이의 상태는 환경 변수와 파일로 전달합니다.


In [ ]:
from pathlib import Path
import hashlib
import os
import tempfile

lab = Path(tempfile.mkdtemp(prefix='bash-security-02-'))
data = lab / 'data'
output = lab / 'output'
data.mkdir()
output.mkdir()
fixtures = {'provenance.txt': 'case_id=COURSE-IR-002\nsource_type=synthetic\nsource_host=lab-web-01\nsource_timezone=Asia/Seoul\nwindow_start=2026-09-10T09:00:00+09:00\nwindow_end=2026-09-10T10:00:00+09:00\ncollector=course-author\ncollection_scope=selected teaching records only\naudit_coverage=partial\nauthorization=offline classroom analysis\n'}
for name, content in fixtures.items():
    path = data / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
before = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
tools_dir = lab / 'tools'
tools_dir.mkdir()
os.environ['COURSE_TOOLS'] = str(tools_dir)
os.environ['COURSE_DATA'] = str(data)
os.environ['COURSE_OUT'] = str(output)
print('합성 자료와 새 결과 폴더 준비 완료')


## Steps

예상 결과: copy_matches=yes, original_and_analysis=separate; 미수집은 0건이 아닙니다.

명령을 실행하기 전에 입력·출력·실패 조건을 표시합니다. 자료의 상세 필드 해석과 정상 행위 대안은 연결된 교안에서 확인합니다.


### 1. 자료의 출처와 시각 확인


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
grep -Fx 'source_type=synthetic' "$COURSE_DATA/provenance.txt"
grep -Fx 'source_timezone=Asia/Seoul' "$COURSE_DATA/provenance.txt"
grep -Fx 'audit_coverage=partial' "$COURSE_DATA/provenance.txt"


### 2. 원본과 작업 사본 구분


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
umask 077
cp "$COURSE_DATA/provenance.txt" "$COURSE_OUT/working-copy.txt"
cmp "$COURSE_DATA/provenance.txt" "$COURSE_OUT/working-copy.txt"
printf 'copy_matches=yes\n'


### 3. 결과를 별도 파일에 기록


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
printf 'case_id=COURSE-IR-002\nanalysis_status=needs_more_evidence\n' > "$COURSE_OUT/analysis.txt"
test "$(wc -l < "$COURSE_OUT/analysis.txt")" -eq 2
printf 'original_and_analysis=separate\n'


### 4. 누락을 정상 결과로 바꾸지 않기


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
if test -r "$COURSE_DATA/not-collected.txt"; then
    printf 'unexpected fixture\n' >&2
    exit 1
else
    printf 'not-collected=unavailable, not zero events\n'
fi


## Checks

각 STEP의 test는 고정 자료의 계산 결과를 검사합니다. 아래는 원본 내용 보존을 확인합니다. 실행 성공과 침해 판정은 다릅니다. 어떤 결과가 사실이고 어떤 결론이 가설인지 교안 질문에 답합니다.


In [ ]:
after = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
assert before == after
print('원본 내용 보존: PASS')
print('분석 결과 파일 수:', sum(p.is_file() for p in output.rglob('*')))


## Next Steps

교안의 완료 기준에 따라 근거·정상 행위 가능성·누락·추가 확인을 제출합니다. 결과는 검토용 임시 폴더에 남습니다. 재실행은 Setup부터 새 폴더에서 시작하며 실제 증거를 공개 저장소에 올리지 않습니다.
